In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
from torchvision import transforms

image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),   # Unified input size
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],   # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ]),
    
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

In [3]:
data_dir = "/kaggle/input/datasets/rishwanthj/xray-ds/split_dataset"

train_dataset = datasets.ImageFolder(
    root=f"{data_dir}/train",
    transform=image_transforms['train']
)
val_dataset = datasets.ImageFolder(
    root=f"{data_dir}/val",
    transform=image_transforms['val']
)
test_dataset = datasets.ImageFolder(
    root=f"{data_dir}/test",
    transform=image_transforms['test']
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [4]:
def get_class_counts(dataset):
    """
    Computes number of samples per class from ImageFolder dataset.
    """
    targets = dataset.targets
    class_to_idx = dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    counts = {}
    for idx in idx_to_class:
        counts[idx] = targets.count(idx)

    return counts


# Compute counts from TRAIN dataset only
class_counts = get_class_counts(train_dataset)

classes = list(class_counts.keys())
counts = list(class_counts.values())

total_samples = sum(counts)
num_classes = len(classes)

weights = []
for i in range(num_classes):
    weight = total_samples / (num_classes * counts[i])
    weights.append(weight)

class_weights_tensor = torch.FloatTensor(weights).to(device)

print("Class counts:", class_counts)
print("Class weights:", weights)

Class counts: {0: 2892, 1: 4809, 2: 8153, 3: 1076}
Class weights: [1.4635200553250345, 0.880120607194843, 0.5191340610818104, 3.933550185873606]


In [5]:
class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)
print("Num classes:", num_classes)

['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Num classes: 4


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

teacher_path = "/kaggle/input/datasets/rishwanthj/efficientnetmini/best_densenet121.pth"

teacher_model = models.efficientnet_b0(weights=None)
in_features = teacher_model.classifier[1].in_features
teacher_model.classifier[1] = nn.Linear(in_features, num_classes)
teacher_model.load_state_dict(torch.load(teacher_path, map_location=device))
teacher_model = teacher_model.to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# TINY CNN MICRO-ARCHITECTURE (EfficientNet Student)
# ==========================================

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)
        return x


class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(16, 32),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(32, 64),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(64, 96),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(96, 128),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 1. Initialize the new TinyCNN Model
model = TinyCNN(num_classes)
model = model.to(device)

print(f"Model Architecture: TinyCNN (Depthwise)")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 2. Setup Criterion and Optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

Model Architecture: TinyCNN (Depthwise)
Total Parameters: 32,484


In [8]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
print(outputs.shape, labels.min().item(), labels.max().item())

torch.Size([32, 4]) 0 3


In [9]:
ce_loss = nn.CrossEntropyLoss(weight=class_weights_tensor)
kl_loss = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [10]:
def train_model_with_early_stopping_distillation(model, teacher_model, optimizer, num_epochs=25, patience=6):
    
    # ===== "Gentle Tutor" Distillation Settings =====
    alpha = 0.7             # 75% Hard Labels (to keep the 90.9% base), 25% Teacher
    temperature = 4.0        # Lower temp keeps teacher predictions sharp
    
    # We use reduction="none" so we can manually apply your class weights to the KD loss
    kl_loss_fn = nn.KLDivLoss(reduction="none")
    ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # ================= TRAIN =================
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # ----- Teacher Forward (No Gradients) -----
            with torch.no_grad():
                teacher_logits = teacher_model(inputs)

            # ----- Student Forward -----
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # ----- Hard Loss (CE) -----
            loss_ce = ce_loss_fn(outputs, labels)

            # ----- Soft Loss (KD) -----
            # Log_softmax for student, standard softmax for teacher
            soft_student = F.log_softmax(outputs / temperature, dim=1)
            soft_teacher = F.softmax(teacher_logits / temperature, dim=1)

            # Calculate KD loss per sample (summed across the 4 classes)
            sample_kd_loss = kl_loss_fn(soft_student, soft_teacher).sum(dim=1) 
            
            # CRITICAL FIX: Multiply the KD loss by the exact same class weights used in CE
            batch_weights = class_weights_tensor[labels]
            weighted_kd_loss = (sample_kd_loss * batch_weights).mean() * (temperature ** 2)

            # ----- Combined Loss -----
            # 75% focus on standard training, 25% focus on teacher's hints
            loss = (1.0 - alpha) * loss_ce + alpha * weighted_kd_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels)

        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = running_corrects.double() / len(train_dataset)

        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc.item()) 

        # ================= VALIDATION =================
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)

                loss = ce_loss_fn(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

        epoch_val_loss = running_loss / len(val_dataset)
        epoch_val_acc = running_corrects.double() / len(val_dataset)

        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc.item()) 

        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}")
        print(f"Val   Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        # ================= EARLY STOPPING =================
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_micro_densenet_distilled.pth")
            print("  --> Best Model Saved!")
        else:
            patience_counter += 1
            print(f"  --> EarlyStopping counter: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    model.load_state_dict(torch.load("best_micro_densenet_distilled.pth"))
    return model, train_losses, val_losses, train_accs, val_accs

In [11]:
model, train_losses, val_losses, train_accs, val_accs = \
    train_model_with_early_stopping_distillation(
        model,
        teacher_model,
        optimizer,
        num_epochs=20,
        patience=5
    )


Epoch 1/20
------------------------------
Train Loss: 4.9396 Acc: 0.4810
Val   Loss: 1.0218 Acc: 0.5735
  --> Best Model Saved!

Epoch 2/20
------------------------------
Train Loss: 3.4676 Acc: 0.5904
Val   Loss: 1.1371 Acc: 0.5943
  --> EarlyStopping counter: 1/5

Epoch 3/20
------------------------------
Train Loss: 2.7388 Acc: 0.6496
Val   Loss: 0.8252 Acc: 0.7035
  --> Best Model Saved!

Epoch 4/20
------------------------------
Train Loss: 2.2486 Acc: 0.7011
Val   Loss: 0.7037 Acc: 0.7551
  --> Best Model Saved!

Epoch 5/20
------------------------------
Train Loss: 1.8675 Acc: 0.7417
Val   Loss: 0.7329 Acc: 0.7768
  --> EarlyStopping counter: 1/5

Epoch 6/20
------------------------------
Train Loss: 1.6287 Acc: 0.7709
Val   Loss: 0.5466 Acc: 0.8128
  --> Best Model Saved!

Epoch 7/20
------------------------------
Train Loss: 1.5002 Acc: 0.7881
Val   Loss: 0.6192 Acc: 0.8019
  --> EarlyStopping counter: 1/5

Epoch 8/20
------------------------------
Train Loss: 1.3913 Acc: 0.8

In [12]:
def evaluate_test_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total
    return test_acc

In [13]:
test_acc = evaluate_test_accuracy(model, test_loader, device)
print(f"Test Accuracy: {test_acc*100:.2f}%")

Test Accuracy: 88.77%


In [14]:
from collections import defaultdict

def per_class_accuracy(model, dataloader, class_names, device):
    model.eval()
    correct = defaultdict(int)
    total = defaultdict(int)

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            for label, pred in zip(labels, preds):
                total[label.item()] += 1
                if label == pred:
                    correct[label.item()] += 1

    for idx, name in enumerate(class_names):
        acc = correct[idx] / total[idx]
        print(f"{name}: {acc * 100:.2f}%")

In [15]:
per_class_accuracy(model, test_loader, class_names, device)

COVID: 94.21%
Lung_Opacity: 88.04%
Normal: 86.18%
Viral Pneumonia: 97.04%


In [16]:
import os

PLOT_DIR = "/kaggle/working/plots"
os.makedirs(PLOT_DIR, exist_ok=True)
import matplotlib.pyplot as plt

def save_loss_plot(train_losses, val_losses):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/loss_curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [17]:
def save_accuracy_plot(train_accs, val_accs):
    plt.figure()
    plt.plot(train_accs, label="Train Accuracy")
    plt.plot(val_accs, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/accuracy_curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [18]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

def save_confusion_matrix(model, dataloader, class_names, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d",
        xticklabels=class_names,
        yticklabels=class_names,
        cmap="Blues"
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.savefig(f"{PLOT_DIR}/confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

In [19]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def save_roc_curves(model, dataloader, class_names, device):
    model.eval()
    y_true, y_scores = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)

            y_true.extend(labels.numpy())
            y_scores.extend(torch.softmax(outputs, dim=1).cpu().numpy())

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

    plt.figure(figsize=(8, 6))
    for i, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_name} (AUC={roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Multi-class ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/roc_curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [20]:
save_loss_plot(train_losses, val_losses)
save_accuracy_plot(train_accs, val_accs)

save_confusion_matrix(
    model,
    test_loader,
    class_names,
    device
)

save_roc_curves(
    model,
    test_loader,
    class_names,
    device
)